In [0]:
%run /Shared/KrPrtnrs_service_principal

In [0]:
%run "./file_utils_nb"

In [0]:
# egress from azure to nifi 
# https://github.com/8451LLC/azure_egress_util
from datetime import datetime

wd = 'abfss://tmkrprtnrs-users@sa8451krprtnrdev.dfs.core.windows.net/l920466/ap-pos-feed'

In [0]:
# combine csv files 

for t in ['competitors']:
  
  files = []
  for i in range(len(dbutils.fs.ls(wd + f"/send"))):
      file = dbutils.fs.ls(wd + '/send')[i][0]
      if t in str(file):
        files.append(file)
  
  df = spark.read.csv(wd + f'/send/*{t}*', header=True)
  
  filename = t + '_' + str(df.agg(f.min('date')).collect()[0][0]) + \
    '_' + str(df.agg(f.max('date')).collect()[0][0])
  
  save_csv(df, directory = wd + '/send', file_name = filename, append_timestamp = True)
    
  for file in files:
    dbutils.fs.rm(file, recurse=True)